# Parameter Counts

We clarify the parameter counts of our models as per the appendix of our report:

- **Backbone params**: the model with the new 5,569 iNatSounds classification head 
removed.
- **With ImageNet-1k head**: total params with the original 1,000 head from ImageNet-1k. 
We use this figure for comparability because it is used in most papers when quoting these 
model's size.
- **With iNatSounds-5569 head**: the full model with the 5,569 iNatSounds head replacing 
the original 1,000 ImageNet head (this is what was actually trained and evaluated).


In [1]:
import torch
import torch.nn as nn
import torchvision
import timm

NUM_CLASSES_INAT = 5569
NUM_CLASSES_IN1K = 1000


def build_model(builder, num_classes):
    if builder == "mobilenet":
        model = torchvision.models.mobilenet_v3_large(weights=None)
        model.classifier[3] = nn.Linear(1280, num_classes)
    elif builder == "efficientnet_b0":
        model = torchvision.models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(1280, num_classes)
    elif builder == "convnext_base":
        model = timm.create_model(
            "convnext_base.fb_in22k_ft_in1k",
            pretrained=False, num_classes=num_classes,
        )
    elif builder == "swin_v2_b":
        model = timm.create_model(
            "swinv2_base_window12to16_192to256.ms_in22k_ft_in1k",
            pretrained=False, num_classes=num_classes,
        )
    else:
        raise ValueError(f"Unknown builder: {builder}")
    return model


def head_params(builder, model):
    """Return the number of parameters in the classification head only."""
    if builder == "mobilenet":
        return sum(p.numel() for p in model.classifier[3].parameters())
    if builder == "efficientnet_b0":
        return sum(p.numel() for p in model.classifier[1].parameters())
    # timm models expose the head via get_classifier()
    return sum(p.numel() for p in model.get_classifier().parameters())


In [ ]:
MODELS = [
    ("MobileNet-V3-Large", "mobilenet"),
    ("EfficientNet-B0", "efficientnet_b0"),
    ("ConvNeXt-Base", "convnext_base"),
    ("Swin-V2-B", "swin_v2_b"),
]

print(f"{'Model':<22} {'Backbone (M)':>14} {'IN-1k head (M)':>16} {'iNat-5569 head (M)':>20}")
print("-" * 76)

for name, builder in MODELS:
    # iNatSounds-head version (what we actually trained / evaluated)
    model_inat = build_model(builder, NUM_CLASSES_INAT)
    total_inat = sum(p.numel() for p in model_inat.parameters())
    head_inat  = head_params(builder, model_inat)
    backbone   = total_inat - head_inat
    del model_inat

    # ImageNet-1k-head version (the figure most papers quote)
    model_in1k = build_model(builder, NUM_CLASSES_IN1K)
    total_in1k = sum(p.numel() for p in model_in1k.parameters())
    del model_in1k

    print(f"{name:<22} {backbone/1e6:>13.2f}M {total_in1k/1e6:>15.2f}M {total_inat/1e6:>19.2f}M")


Model                    Backbone (M)   IN-1k head (M)   iNat-5569 head (M)
----------------------------------------------------------------------------
MobileNet-V3-Large              4.20M            5.48M               11.34M
EfficientNet-B0                 4.01M            5.29M               11.14M
ConvNeXt-Base                  87.57M           88.59M               93.27M
Swin-V2-B                      86.89M           87.92M               92.60M
